In [ ]:
from pathlib import Path
import importlib
import importlib.metadata as bootstrap_metadata
import importlib.util
import os
import subprocess
import sys

CPU_VALIDATE = os.environ.get("ANOXFUSE_CPU_VALIDATE", "0") == "1"
TRANSFORMERS_ALREADY_IMPORTED = "transformers" in sys.modules

CONFIG = {
    "seed": 70877,
    "run_gpu": not CPU_VALIDATE,
    "force_retrain": False,
    "n_splits": 3 if CPU_VALIDATE else 5,
    "similarity_cutoff": 60.0,
    "lgbm_trees": 120 if CPU_VALIDATE else 500,
    "esm2_batch_size": 64 if CPU_VALIDATE else 256,
}
ESM2_PEPTIDE_ID = "jiahuizhang/esm-150m-peptide-fine-tune"
ESM2_PEPTIDE_REV = "6d8cebf"

TORCH_PRESENT = importlib.util.find_spec("torch") is not None
if CONFIG["run_gpu"] and not TORCH_PRESENT:
    raise RuntimeError(
        "PyTorch is absent. Select a CUDA/PyTorch GPU image. "
        "This notebook intentionally never installs or replaces torch."
    )

CORE_REQUIRED = {
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.1",
    "scipy": "scipy>=1.11",
    "sklearn": "scikit-learn>=1.4",
    "requests": "requests>=2.31",
    "rapidfuzz": "rapidfuzz>=3.6",
    "rdkit": "rdkit>=2024.3.1",
    "lightgbm": "lightgbm>=4.3",
    "tqdm": "tqdm>=4.66",
}
GPU_REQUIRED = {
    "transformers": "transformers==5.14.1",
    "accelerate": "accelerate>=1.1",
    "huggingface_hub": "huggingface_hub>=0.26",
    "tokenizers": "tokenizers>=0.20",
    "safetensors": "safetensors>=0.4",
}
REQUIRED = {**CORE_REQUIRED, **(GPU_REQUIRED if CONFIG["run_gpu"] else {})}

def dependency_needs_install(module, requirement):
    if importlib.util.find_spec(module) is None:
        return True
    if "==" in requirement:
        package, required_version = requirement.split("==", 1)
        try:
            return bootstrap_metadata.version(package) != required_version
        except bootstrap_metadata.PackageNotFoundError:
            return True
    return False

missing = [
    requirement
    for module, requirement in REQUIRED.items()
    if dependency_needs_install(module, requirement)
]
if missing:
    print("Installing missing non-PyTorch packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    importlib.invalidate_caches()
    if TRANSFORMERS_ALREADY_IMPORTED and any(
        requirement.startswith("transformers==") for requirement in missing
    ):
        raise RuntimeError(
            "Transformers was already imported before its version was repaired. "
            "Restart the kernel now and run again from the first cell."
        )
else:
    print("All non-PyTorch dependencies are available.")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Configuration:", CONFIG)


In [ ]:
import gc
import hashlib
import importlib.metadata as metadata
import json
import math
import random
import re
import shutil
import sys
import warnings
import zipfile

import numpy as np
import pandas as pd
import joblib
import requests
from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)

if TORCH_PRESENT:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    CUDA_READY = bool(torch.cuda.is_available() and torch.version.cuda)
else:
    torch = None
    CUDA_READY = False

if CONFIG["run_gpu"] and not CUDA_READY:
    raise RuntimeError(
        "CUDA is unavailable. Stop this rental and choose a CUDA-enabled PyTorch image "
        "before any model work is attempted."
    )
ENABLE_GPU = bool(CONFIG["run_gpu"] and CUDA_READY)
FREE_DISK_GIB = shutil.disk_usage(Path.cwd()).free / (1024 ** 3)
if ENABLE_GPU and FREE_DISK_GIB < 8:
    raise RuntimeError(
        f"Only {FREE_DISK_GIB:.1f} GiB is free. Allocate at least 8 GiB for the "
        "150M checkpoint, package/cache staging, embeddings, and model artifacts."
    )

ROOT = Path.cwd() / "anoxfuse_run"
DATA_DIR = ROOT / "data"
CACHE_DIR = ROOT / "cache"
ARTIFACT_DIR = ROOT / "artifacts"
TABLE_DIR = ROOT / "tables"
for directory in (ROOT, DATA_DIR, CACHE_DIR, ARTIFACT_DIR, TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def cleanup_gpu():
    gc.collect()
    if TORCH_PRESENT and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def package_version(name):
    try:
        return metadata.version(name)
    except Exception:
        return None

if ENABLE_GPU:
    from transformers import AutoModel, AutoTokenizer
    print("Running an early CUDA smoke test for", ESM2_PEPTIDE_ID)
    smoke_tokenizer = AutoTokenizer.from_pretrained(
        ESM2_PEPTIDE_ID, revision=ESM2_PEPTIDE_REV
    )
    smoke_model = AutoModel.from_pretrained(
        ESM2_PEPTIDE_ID, revision=ESM2_PEPTIDE_REV
    ).to("cuda").eval()
    smoke_tokens = {
        key: value.to("cuda")
        for key, value in smoke_tokenizer(
            ["ACDEFGHIK"], return_tensors="pt", padding=True
        ).items()
    }
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    with torch.inference_mode(), torch.amp.autocast("cuda", dtype=amp_dtype):
        smoke_hidden = smoke_model(**smoke_tokens).last_hidden_state
    assert smoke_hidden.ndim == 3 and torch.isfinite(smoke_hidden).all()
    print(
        "CUDA smoke test passed:",
        torch.cuda.get_device_name(0),
        tuple(smoke_hidden.shape),
        smoke_hidden.dtype,
    )
    del smoke_hidden, smoke_tokens, smoke_model, smoke_tokenizer
    cleanup_gpu()

MANIFEST = {
    "python": sys.version,
    "torch": package_version("torch"),
    "transformers": package_version("transformers"),
    "cuda_ready": CUDA_READY,
    "cuda_version": getattr(getattr(torch, "version", None), "cuda", None),
    "gpu": torch.cuda.get_device_name(0) if CUDA_READY else None,
    "free_disk_gib_at_start": round(FREE_DISK_GIB, 2),
    "model_id": ESM2_PEPTIDE_ID,
    "model_revision": ESM2_PEPTIDE_REV,
    "config": CONFIG,
}
(ARTIFACT_DIR / "environment_manifest.json").write_text(
    json.dumps(MANIFEST, indent=2, default=str), encoding="utf-8"
)
print(json.dumps(MANIFEST, indent=2, default=str))


In [ ]:
GITHUB_REPOSITORY = "Offpass/AnOxFuse"
GITHUB_BRANCH = os.environ.get("ANOXFUSE_GITHUB_BRANCH", "main")
RAW_DATA_BASE = (
    f"https://raw.githubusercontent.com/{GITHUB_REPOSITORY}/"
    f"{GITHUB_BRANCH}/data"
)
REPOSITORY_DATA_DIR = Path.cwd() / "data"
DATA_FILENAMES = (
    "remaining_positive.fasta",
    "remaining_negative.fasta",
    "independent_test_cleaned.fasta",
)

def materialize_data_file(filename):
    target = DATA_DIR / filename
    local = REPOSITORY_DATA_DIR / filename
    if target.exists() and target.stat().st_size > 0:
        return target
    if local.exists() and local.stat().st_size > 0:
        shutil.copy2(local, target)
        return target
    url = f"{RAW_DATA_BASE}/{filename}"
    response = requests.get(url, timeout=120)
    if response.status_code == 404:
        raise RuntimeError(
            f"{filename} is not available at {url}. "
            f"Add the dataset to the data folder on the {GITHUB_BRANCH} branch."
        )
    response.raise_for_status()
    temporary = target.with_suffix(target.suffix + ".partial")
    temporary.write_bytes(response.content)
    if temporary.stat().st_size == 0:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(f"Downloaded an empty data file from {url}")
    temporary.replace(target)
    return target

DATA_PATHS = {
    filename: materialize_data_file(filename)
    for filename in DATA_FILENAMES
}

STANDARD_AA = "ACDEFGHIKLMNPQRSTVWY"

def read_fasta(path):
    records, header, chunks = [], None, []
    with open(path, encoding="utf-8") as handle:
        for raw in handle:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(chunks).upper()))
                header, chunks = line[1:], []
            else:
                chunks.append(line)
    if header is not None:
        records.append((header, "".join(chunks).upper()))
    return records

def label_from_header(header):
    if header and header[0] in "01":
        return int(header[0])
    match = re.search(r"(?:^|\|)([01])(?:$|\|)", header)
    if match:
        return int(match.group(1))
    raise ValueError(f"Cannot infer label from header: {header}")

def dataframe_from_records(records, labels=None, source=""):
    if labels is None:
        labels = [label_from_header(header) for header, _ in records]
    frame = pd.DataFrame({
        "header": [header for header, _ in records],
        "sequence": [sequence for _, sequence in records],
        "label": np.asarray(labels, dtype=int),
        "source": source,
    })
    frame["length"] = frame["sequence"].str.len()
    frame["valid"] = frame["sequence"].map(
        lambda sequence: bool(sequence) and set(sequence) <= set(STANDARD_AA)
    )
    return frame

positive_records = read_fasta(DATA_PATHS["remaining_positive.fasta"])
negative_records = read_fasta(DATA_PATHS["remaining_negative.fasta"])
test_records = read_fasta(DATA_PATHS["independent_test_cleaned.fasta"])

DEV = pd.concat([
    dataframe_from_records(
        positive_records,
        [1] * len(positive_records),
        "remaining_positive",
    ),
    dataframe_from_records(
        negative_records,
        [0] * len(negative_records),
        "remaining_negative",
    ),
], ignore_index=True)
TEST = dataframe_from_records(
    test_records,
    source="published_independent_test",
)

assert DEV["valid"].all() and TEST["valid"].all()
assert len(DEV) == 2735 and len(TEST) == 302
assert tuple(TEST["label"].value_counts().sort_index()) == (152, 150)

DATASET_SUMMARY = pd.DataFrame([
    {
        "set": "development",
        "n": len(DEV),
        "positive": int(DEV.label.sum()),
        "negative": int((1 - DEV.label).sum()),
        "mean_length": DEV.length.mean(),
    },
    {
        "set": "released independent test",
        "n": len(TEST),
        "positive": int(TEST.label.sum()),
        "negative": int((1 - TEST.label).sum()),
        "mean_length": TEST.length.mean(),
    },
])
DATASET_SUMMARY.to_csv(TABLE_DIR / "dataset_summary.csv", index=False)
print(DATASET_SUMMARY.round(3).to_string(index=False))
print("Exact development/test overlaps:", len(set(DEV.sequence) & set(TEST.sequence)))


In [ ]:
from rapidfuzz import fuzz, process

def sequence_hash(sequences):
    digest = hashlib.sha256()
    for sequence in sequences:
        digest.update(str(sequence).encode())
        digest.update(b"\0")
    return digest.hexdigest()[:16]

def load_cached_X(cache):
    if not cache.exists():
        return None
    try:
        with np.load(cache) as payload:
            return payload["X"]
    except (OSError, ValueError, EOFError, KeyError, zipfile.BadZipFile) as exc:
        print("Removing unreadable cache:", cache.name, repr(exc))
        cache.unlink(missing_ok=True)
        return None

def save_cached_X(cache, X):
    temporary = cache.with_name(cache.name + ".incomplete")
    try:
        with open(temporary, "wb") as handle:
            np.savez_compressed(handle, X=X)
        temporary.replace(cache)
    finally:
        temporary.unlink(missing_ok=True)

def greedy_similarity_groups(sequences, cutoff=60.0):
    sequences = list(sequences)
    cache = CACHE_DIR / f"groups_{sequence_hash(sequences)}_{int(cutoff)}.npy"
    if cache.exists():
        return np.load(cache)
    similarity = process.cdist(
        sequences,
        sequences,
        scorer=fuzz.ratio,
        score_cutoff=cutoff,
        dtype=np.uint8,
        workers=-1,
    )
    groups = np.full(len(sequences), -1, dtype=int)
    order = sorted(
        range(len(sequences)),
        key=lambda index: (-len(sequences[index]), sequences[index], index),
    )
    group_id = 0
    for representative in order:
        if groups[representative] != -1:
            continue
        eligible = (groups == -1) & (similarity[representative] >= cutoff)
        groups[eligible] = group_id
        group_id += 1
    np.save(cache, groups)
    return groups

DEV_GROUPS = greedy_similarity_groups(
    DEV.sequence, CONFIG["similarity_cutoff"]
)
fold_builder = StratifiedGroupKFold(
    n_splits=CONFIG["n_splits"],
    shuffle=True,
    random_state=SEED,
)
MAIN_FOLDS = list(
    fold_builder.split(DEV.sequence, DEV.label, groups=DEV_GROUPS)
)
FOLD_ID = np.full(len(DEV), -1, dtype=int)
fold_audit = []
for fold, (train_index, valid_index) in enumerate(MAIN_FOLDS):
    FOLD_ID[valid_index] = fold
    fold_audit.append({
        "fold": fold + 1,
        "train_n": len(train_index),
        "valid_n": len(valid_index),
        "valid_positive_rate": DEV.label.iloc[valid_index].mean(),
        "train_groups": len(set(DEV_GROUPS[train_index])),
        "valid_groups": len(set(DEV_GROUPS[valid_index])),
    })
FOLD_AUDIT = pd.DataFrame(fold_audit)
assert np.all(FOLD_ID >= 0)
assert sum(len(valid) for _, valid in MAIN_FOLDS) == len(DEV)
FOLD_AUDIT.to_csv(TABLE_DIR / "grouped_fold_audit.csv", index=False)
print(FOLD_AUDIT.round(3).to_string(index=False))
print("Similarity groups:", len(np.unique(DEV_GROUPS)))
print("Development samples retained:", len(DEV), "/", len(DEV))


In [ ]:
from lightgbm import LGBMClassifier
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

EPS = 1e-6
ECFP4_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048
)

def safe_auc(y, probability):
    y = np.asarray(y)
    probability = np.asarray(probability, dtype=float)
    return roc_auc_score(y, probability) if len(np.unique(y)) == 2 else np.nan

def binary_metrics(y, probability, threshold=0.5):
    y = np.asarray(y, dtype=int)
    raw = np.asarray(probability, dtype=float)
    clipped = np.clip(raw, EPS, 1 - EPS)
    prediction = (raw >= threshold).astype(int)
    return {
        "AUC": safe_auc(y, raw),
        "AUPRC": average_precision_score(y, raw),
        "ACC": accuracy_score(y, prediction),
        "F1": f1_score(y, prediction, zero_division=0),
        "Precision": precision_score(y, prediction, zero_division=0),
        "Recall": recall_score(y, prediction, zero_division=0),
        "MCC": matthews_corrcoef(y, prediction),
        "Brier": brier_score_loss(y, clipped),
        "LogLoss": log_loss(y, clipped, labels=[0, 1]),
    }

def fingerprint_matrix(sequences):
    sequences = list(sequences)
    cache = CACHE_DIR / f"ecfp4_count_{sequence_hash(sequences)}.npz"
    cached = load_cached_X(cache)
    if cached is not None:
        return cached
    rows = []
    for sequence in tqdm(sequences, desc="ECFP4-count fingerprints"):
        molecule = Chem.MolFromFASTA(sequence)
        if molecule is None:
            raise ValueError(f"RDKit could not construct peptide: {sequence}")
        rows.append(
            ECFP4_GENERATOR.GetCountFingerprintAsNumPy(molecule).astype(np.float32)
        )
    X = np.vstack(rows)
    save_cached_X(cache, X)
    return X

def lgbm_factory(fold=0):
    return LGBMClassifier(
        n_estimators=CONFIG["lgbm_trees"],
        random_state=SEED + fold,
        n_jobs=-1,
        verbosity=-1,
    )

def logistic_factory(_fold=0):
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            C=1.0,
            max_iter=4000,
            solver="liblinear",
            random_state=SEED,
        )),
    ])

def cv_tabular(name, X, y, folds, factory):
    y = np.asarray(y, dtype=int)
    oof = np.full(len(y), np.nan)
    rows = []
    for fold, (train_index, valid_index) in enumerate(folds):
        model = factory(fold).fit(X[train_index], y[train_index])
        probability = model.predict_proba(X[valid_index])[:, 1]
        oof[valid_index] = probability
        rows.append({
            "model": name,
            "fold": fold + 1,
            **binary_metrics(y[valid_index], probability),
        })
    assert np.isfinite(oof).all()
    return oof, pd.DataFrame(rows)

def hf_esm2_embeddings(sequences, cache_tag):
    if not ENABLE_GPU:
        raise RuntimeError("Frozen ESM2 embedding extraction requires CUDA.")
    sequences = list(sequences)
    cache = CACHE_DIR / (
        f"peptide_esm2_{ESM2_PEPTIDE_REV}_{cache_tag}_"
        f"{sequence_hash(sequences)}_meanmax.npz"
    )
    cached = load_cached_X(cache)
    if cached is not None:
        return cached

    from transformers import AutoModel, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        ESM2_PEPTIDE_ID, revision=ESM2_PEPTIDE_REV
    )
    model = AutoModel.from_pretrained(
        ESM2_PEPTIDE_ID, revision=ESM2_PEPTIDE_REV
    ).to("cuda").eval()
    special_ids = torch.tensor(
        tokenizer.all_special_ids, dtype=torch.long, device="cuda"
    )
    rows = []
    batch_size = CONFIG["esm2_batch_size"]
    start = 0
    while start < len(sequences):
        batch = sequences[start:start + batch_size]
        try:
            tokens = tokenizer(batch, return_tensors="pt", padding=True)
            tokens = {key: value.to("cuda") for key, value in tokens.items()}
            input_ids = tokens["input_ids"]
            attention = tokens["attention_mask"].bool()
            special = (input_ids.unsqueeze(-1) == special_ids).any(-1)
            residue_mask = attention & ~special
            amp_dtype = (
                torch.bfloat16
                if torch.cuda.is_bf16_supported()
                else torch.float16
            )
            with torch.inference_mode(), torch.amp.autocast(
                "cuda", dtype=amp_dtype
            ):
                hidden = model(**tokens).last_hidden_state
            mean = (
                hidden * residue_mask.unsqueeze(-1)
            ).sum(1) / residue_mask.sum(1, keepdim=True)
            maximum = hidden.masked_fill(
                ~residue_mask.unsqueeze(-1), -torch.inf
            ).max(1).values
            rows.append(
                torch.cat([mean, maximum], dim=1).float().cpu().numpy()
            )
            start += len(batch)
        except torch.cuda.OutOfMemoryError:
            cleanup_gpu()
            batch_size //= 2
            if batch_size < 1:
                raise
            print("Reduced ESM2 batch size to", batch_size)
    X = np.vstack(rows)
    save_cached_X(cache, X)
    del model, tokenizer
    cleanup_gpu()
    return X

def nested_anoxfuse(X_local, X_context, y, groups, outer_folds):
    y = np.asarray(y, dtype=int)
    groups = np.asarray(groups)
    probability = np.full(len(y), np.nan)
    metric_rows = []
    coefficient_rows = []

    for outer_fold, (outer_train, outer_valid) in enumerate(outer_folds):
        inner_splitter = StratifiedGroupKFold(
            n_splits=4,
            shuffle=True,
            random_state=SEED + 1000 + outer_fold,
        )
        inner_folds = list(
            inner_splitter.split(
                np.zeros(len(outer_train)),
                y[outer_train],
                groups=groups[outer_train],
            )
        )
        inner_scores = []
        outer_scores = []
        for X, factory in (
            (X_local, lgbm_factory),
            (X_context, logistic_factory),
        ):
            inner_oof = np.full(len(outer_train), np.nan)
            for inner_fold, (inner_train_rel, inner_valid_rel) in enumerate(
                inner_folds
            ):
                inner_train = outer_train[inner_train_rel]
                inner_valid = outer_train[inner_valid_rel]
                model = factory(
                    10_000 * outer_fold + inner_fold
                ).fit(X[inner_train], y[inner_train])
                inner_oof[inner_valid_rel] = model.predict_proba(
                    X[inner_valid]
                )[:, 1]
            assert np.isfinite(inner_oof).all()
            outer_model = factory(20_000 + outer_fold).fit(
                X[outer_train], y[outer_train]
            )
            inner_scores.append(inner_oof)
            outer_scores.append(
                outer_model.predict_proba(X[outer_valid])[:, 1]
            )

        inner_Z = np.column_stack([
            logit(np.clip(score, EPS, 1 - EPS))
            for score in inner_scores
        ])
        outer_Z = np.column_stack([
            logit(np.clip(score, EPS, 1 - EPS))
            for score in outer_scores
        ])
        meta = LogisticRegression(
            C=1.0, solver="lbfgs", max_iter=2000
        ).fit(inner_Z, y[outer_train])
        fold_probability = meta.predict_proba(outer_Z)[:, 1]
        probability[outer_valid] = fold_probability
        metric_rows.append({
            "fold": outer_fold + 1,
            **binary_metrics(y[outer_valid], fold_probability),
        })
        coefficient_rows.append({
            "fold": outer_fold + 1,
            "Local chemistry score": float(meta.coef_[0, 0]),
            "Contextual sequence score": float(meta.coef_[0, 1]),
            "Intercept": float(meta.intercept_[0]),
        })

    assert np.isfinite(probability).all()
    return (
        probability,
        pd.DataFrame(metric_rows),
        pd.DataFrame(coefficient_rows),
    )


In [ ]:
BUNDLE_PATH = ARTIFACT_DIR / "anoxfuse_main_run.npz"
FOLD_METRICS_PATH = TABLE_DIR / "anoxfuse_grouped_fold_metrics.csv"
COEFFICIENT_PATH = TABLE_DIR / "anoxfuse_fusion_coefficients.csv"
LOCAL_MODEL_PATH = ARTIFACT_DIR / "anoxfuse_local_model.joblib"
CONTEXT_MODEL_PATH = ARTIFACT_DIR / "anoxfuse_context_model.joblib"
META_MODEL_PATH = ARTIFACT_DIR / "anoxfuse_meta_model.joblib"
CACHE_COMPLETE = all(path.exists() for path in (
    BUNDLE_PATH,
    FOLD_METRICS_PATH,
    COEFFICIENT_PATH,
    LOCAL_MODEL_PATH,
    CONTEXT_MODEL_PATH,
    META_MODEL_PATH,
))

if CACHE_COMPLETE and not CONFIG["force_retrain"]:
    print("Loading cached AnOxFuse main-run bundle:", BUNDLE_PATH)
    with np.load(BUNDLE_PATH) as payload:
        DEV_Y = payload["dev_y"]
        TEST_Y = payload["test_y"]
        ANOXFUSE_OOF = payload["anoxfuse_oof"]
        ANOXFUSE_TEST = payload["anoxfuse_test"]
        LOCAL_OOF = payload["local_oof"]
        LOCAL_TEST = payload["local_test"]
        CONTEXT_OOF = payload["context_oof"]
        CONTEXT_TEST = payload["context_test"]
        FINAL_META_COEF = payload["final_meta_coef"]
        FINAL_META_INTERCEPT = float(payload["final_meta_intercept"][0])
    FOLD_METRICS = pd.read_csv(FOLD_METRICS_PATH)
    META_COEFFICIENTS = pd.read_csv(COEFFICIENT_PATH)
    local_model = joblib.load(LOCAL_MODEL_PATH)
    context_model = joblib.load(CONTEXT_MODEL_PATH)
    final_meta = joblib.load(META_MODEL_PATH)
else:
    if not ENABLE_GPU:
        raise RuntimeError(
            "No cached main run exists. Use the rented NVIDIA device to create it."
        )
    DEV_Y = DEV.label.to_numpy(dtype=int)
    TEST_Y = TEST.label.to_numpy(dtype=int)

    print("Building local-chemistry representations...")
    DEV_LOCAL = fingerprint_matrix(DEV.sequence)
    TEST_LOCAL = fingerprint_matrix(TEST.sequence)

    print("Extracting the only contextual representation used by AnOxFuse...")
    DEV_CONTEXT = hf_esm2_embeddings(DEV.sequence, "development")
    TEST_CONTEXT = hf_esm2_embeddings(TEST.sequence, "released_test")

    print("Cross-fitting both main branches...")
    LOCAL_OOF, _ = cv_tabular(
        "Local chemistry score",
        DEV_LOCAL,
        DEV_Y,
        MAIN_FOLDS,
        lgbm_factory,
    )
    CONTEXT_OOF, _ = cv_tabular(
        "Contextual sequence score",
        DEV_CONTEXT,
        DEV_Y,
        MAIN_FOLDS,
        logistic_factory,
    )

    local_model = lgbm_factory(0).fit(DEV_LOCAL, DEV_Y)
    context_model = logistic_factory(0).fit(DEV_CONTEXT, DEV_Y)
    LOCAL_TEST = local_model.predict_proba(TEST_LOCAL)[:, 1]
    CONTEXT_TEST = context_model.predict_proba(TEST_CONTEXT)[:, 1]

    print("Running leakage-safe nested fusion...")
    ANOXFUSE_OOF, FOLD_METRICS, META_COEFFICIENTS = nested_anoxfuse(
        DEV_LOCAL,
        DEV_CONTEXT,
        DEV_Y,
        DEV_GROUPS,
        MAIN_FOLDS,
    )

    development_Z = np.column_stack([
        logit(np.clip(LOCAL_OOF, EPS, 1 - EPS)),
        logit(np.clip(CONTEXT_OOF, EPS, 1 - EPS)),
    ])
    test_Z = np.column_stack([
        logit(np.clip(LOCAL_TEST, EPS, 1 - EPS)),
        logit(np.clip(CONTEXT_TEST, EPS, 1 - EPS)),
    ])
    final_meta = LogisticRegression(
        C=1.0, solver="lbfgs", max_iter=2000
    ).fit(development_Z, DEV_Y)
    ANOXFUSE_TEST = final_meta.predict_proba(test_Z)[:, 1]
    FINAL_META_COEF = final_meta.coef_[0].astype(float)
    FINAL_META_INTERCEPT = float(final_meta.intercept_[0])

    joblib.dump(local_model, LOCAL_MODEL_PATH)
    joblib.dump(context_model, CONTEXT_MODEL_PATH)
    joblib.dump(final_meta, META_MODEL_PATH)

    np.savez_compressed(
        BUNDLE_PATH,
        dev_y=DEV_Y,
        test_y=TEST_Y,
        anoxfuse_oof=ANOXFUSE_OOF,
        anoxfuse_test=ANOXFUSE_TEST,
        local_oof=LOCAL_OOF,
        local_test=LOCAL_TEST,
        context_oof=CONTEXT_OOF,
        context_test=CONTEXT_TEST,
        final_meta_coef=FINAL_META_COEF,
        final_meta_intercept=np.array([FINAL_META_INTERCEPT]),
    )
    FOLD_METRICS.to_csv(FOLD_METRICS_PATH, index=False)
    META_COEFFICIENTS.to_csv(COEFFICIENT_PATH, index=False)

assert len(ANOXFUSE_OOF) == len(DEV)
assert len(ANOXFUSE_TEST) == len(TEST)
assert np.isfinite(ANOXFUSE_OOF).all() and np.isfinite(ANOXFUSE_TEST).all()

OOF_METRICS = binary_metrics(DEV_Y, ANOXFUSE_OOF)
TEST_METRICS = binary_metrics(TEST_Y, ANOXFUSE_TEST)
MAIN_SUMMARY = pd.DataFrame([
    {"evaluation": "Similarity-grouped development OOF", **OOF_METRICS},
    {"evaluation": "Released independent test", **TEST_METRICS},
])
MAIN_SUMMARY.to_csv(TABLE_DIR / "anoxfuse_main_summary.csv", index=False)

OOF_EXPORT = DEV[["header", "sequence", "length", "label"]].copy()
OOF_EXPORT["fold"] = FOLD_ID + 1
OOF_EXPORT["similarity_group"] = DEV_GROUPS
OOF_EXPORT["probability"] = ANOXFUSE_OOF
OOF_EXPORT["prediction"] = (ANOXFUSE_OOF >= 0.5).astype(int)
OOF_EXPORT["confidence"] = 2 * np.abs(ANOXFUSE_OOF - 0.5)
OOF_EXPORT.to_csv(TABLE_DIR / "anoxfuse_oof_predictions.csv", index=False)

TEST_EXPORT = TEST[["header", "sequence", "length", "label"]].copy()
TEST_EXPORT["probability"] = ANOXFUSE_TEST
TEST_EXPORT["prediction"] = (ANOXFUSE_TEST >= 0.5).astype(int)
TEST_EXPORT["confidence"] = 2 * np.abs(ANOXFUSE_TEST - 0.5)
TEST_EXPORT.to_csv(TABLE_DIR / "anoxfuse_test_predictions.csv", index=False)

print(MAIN_SUMMARY.round(4).to_string(index=False))
print("Saved model artifacts to", ARTIFACT_DIR)
print("Saved prediction tables to", TABLE_DIR)


In [ ]:
def predict_anoxfuse(sequences, threshold=0.5):
    sequences = [str(sequence).strip().upper() for sequence in sequences]
    invalid = [
        sequence
        for sequence in sequences
        if not sequence or not set(sequence) <= set(STANDARD_AA)
    ]
    if invalid:
        raise ValueError(f"Invalid peptide sequences: {invalid[:3]}")
    local_features = fingerprint_matrix(sequences)
    context_features = hf_esm2_embeddings(
        sequences,
        f"inference_{sequence_hash(sequences)}",
    )
    local_probability = local_model.predict_proba(local_features)[:, 1]
    context_probability = context_model.predict_proba(context_features)[:, 1]
    fusion_features = np.column_stack([
        logit(np.clip(local_probability, EPS, 1 - EPS)),
        logit(np.clip(context_probability, EPS, 1 - EPS)),
    ])
    probability = final_meta.predict_proba(fusion_features)[:, 1]
    return pd.DataFrame({
        "sequence": sequences,
        "local_probability": local_probability,
        "context_probability": context_probability,
        "anoxfuse_probability": probability,
        "prediction": (probability >= threshold).astype(int),
    })
